# Лабораторная 07. shuffle partitions и AQE

Цель: увидеть, как `spark.sql.shuffle.partitions` и AQE влияют на количество post-shuffle tasks.

In [2]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('lab-07-aqe').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Spark UI: http://0a370e2ebe67:4040


## Часть 1. AQE выключен, shuffle partitions = 200
После action посмотрите stage после shuffle. Ожидайте около 200 reduce tasks.

In [3]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '200')
q_200 = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_200.explain('formatted')
q_200.count()

== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_id#3L]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<customer_id:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [customer_id#3L]

(3) HashAggregate [codegen id : 1]
Input [1]: [customer_id#3L]
Keys [1]: [customer_id#3L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#30L]
Results [2]: [customer_id#3L, count#31L]

(4) Exchange
Input [2]: [customer_id#3L, count#31L]
Arguments: hashpartitioning(customer_id#3L, 200), ENSURE_REQUIREMENTS, [plan_id=20]

(5) HashAggregate [codegen id : 2]
Input [2]: [customer_id#3L, count#31L]
Keys [1]: [customer_id#3L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#25L]
Results [2]: [customer_id#3L, count(1)#25L AS orders_cnt#26L]




10000

## Часть 2. AQE выключен, shuffle partitions = 8

In [4]:
spark.conf.set('spark.sql.adaptive.enabled', 'false')
spark.conf.set('spark.sql.shuffle.partitions', '8')
q_8 = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_8.explain('formatted')
q_8.count()

== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * ColumnarToRow (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_id#3L]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<customer_id:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [1]: [customer_id#3L]

(3) HashAggregate [codegen id : 1]
Input [1]: [customer_id#3L]
Keys [1]: [customer_id#3L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#48L]
Results [2]: [customer_id#3L, count#49L]

(4) Exchange
Input [2]: [customer_id#3L, count#49L]
Arguments: hashpartitioning(customer_id#3L, 8), ENSURE_REQUIREMENTS, [plan_id=110]

(5) HashAggregate [codegen id : 2]
Input [2]: [customer_id#3L, count#49L]
Keys [1]: [customer_id#3L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#44L]
Results [2]: [customer_id#3L, count(1)#44L AS orders_cnt#45L]




10000

## Часть 3. AQE включен, initial shuffle partitions = 200
AQE может объединить маленькие post-shuffle partitions. Смотрите SQL tab: adaptive plan и фактическое количество tasks.

In [5]:
spark.conf.set('spark.sql.adaptive.enabled', 'true')
spark.conf.set('spark.sql.shuffle.partitions', '200')
spark.conf.set('spark.sql.adaptive.coalescePartitions.enabled', 'true')
spark.conf.set('spark.sql.adaptive.advisoryPartitionSizeInBytes', '16m')
q_aqe = orders.groupBy('customer_id').agg(F.count('*').alias('orders_cnt'))
q_aqe.explain('formatted')
q_aqe.count()
q_aqe.explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_id#3L]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<customer_id:bigint>

(2) HashAggregate
Input [1]: [customer_id#3L]
Keys [1]: [customer_id#3L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#66L]
Results [2]: [customer_id#3L, count#67L]

(3) Exchange
Input [2]: [customer_id#3L, count#67L]
Arguments: hashpartitioning(customer_id#3L, 200), ENSURE_REQUIREMENTS, [plan_id=191]

(4) HashAggregate
Input [2]: [customer_id#3L, count#67L]
Keys [1]: [customer_id#3L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#62L]
Results [2]: [customer_id#3L, count(1)#62L AS orders_cnt#63L]

(5) AdaptiveSparkPlan
Output [2]: [customer_id#3L, orders_cnt#63L]
Arguments: isFinalPlan=false




[Stage 8:============================================>              (3 + 1) / 4]

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [customer_id#3L]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
ReadSchema: struct<customer_id:bigint>

(2) HashAggregate
Input [1]: [customer_id#3L]
Keys [1]: [customer_id#3L]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#66L]
Results [2]: [customer_id#3L, count#67L]

(3) Exchange
Input [2]: [customer_id#3L, count#67L]
Arguments: hashpartitioning(customer_id#3L, 200), ENSURE_REQUIREMENTS, [plan_id=191]

(4) HashAggregate
Input [2]: [customer_id#3L, count#67L]
Keys [1]: [customer_id#3L]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#62L]
Results [2]: [customer_id#3L, count(1)#62L AS orders_cnt#63L]

(5) AdaptiveSparkPlan
Output [2]: [customer_id#3L, orders_cnt#63L]
Arguments: isFinalPlan=false




Заполните:

| Часть | AQE | `spark.sql.shuffle.partitions` | JOB ID | Tasks после shuffle в Spark UI | Duration | Что изменилось |
|---|---|---:|---:|---|---|
| 1 | false | 2 | 200 | 200 | 31s | Много задач из-за 200 партиций |
| 2 | false | 3 | 8 | 8 | ? | 0.4s | Мало задач из-за 8 партиций |
| 3 | true | 4,5,6 | 4 (job 4) | 1+1 | 2s | AQE объединил 200→4 партиции |

Вопросы:

- Сколько partitions было задано изначально? 3 колонка
- Сколько реально получилось? 4 колонка
- Почему AQE может уменьшить количество маленьких tasks? потому что spark.sql.adaptive.advisoryPartitionSizeInBytes задает ему ориентир в 16 МБ и AQE объединяет несколько маленьких партиций в одну с этим целевым размером 

In [6]:
spark.stop()